In [ ]:
import pandas as pd
import numpy as np
from scipy.io.wavfile import write
from tqdm import tqdm
import matplotlib.pyplot as plt
from scipy.signal import spectrogram

# Read the EEG data from a CSV file
data = pd.read_csv('data/eeg_data.csv', low_memory=False)  # Use low_memory=False to avoid dtype warnings
data['TimeStamp'] = pd.to_datetime(data['TimeStamp'])
data.set_index('TimeStamp', inplace=True)

# Combine Delta columns into a single column for the full dataset
data['Delta_combined'] = data[['Delta_TP9', 'Delta_AF7', 'Delta_AF8', 'Delta_TP10']].sum(axis=1)

# Calculate the mean of the Delta combined column in minute intervals for the full dataset
minute_resampled = data.index.to_period('Min')
data['minute_resampled'] = data.groupby(minute_resampled)['Delta_combined'].transform('mean')

# Define frequencies for C major scale (do-re-mi-fa-sol-la-ti-do)
scale = np.array([261.63, 293.66, 329.63, 349.23, 392.00, 440.00, 493.88, 523.25])  # Frequencies in Hz

# Map EEG data continuously to musical notes in the C major scale for the full dataset
mapped_indices = np.interp(data['minute_resampled'], 
                            (data['minute_resampled'].min(), data['minute_resampled'].max()), 
                            (0, len(scale) - 1))  # Map to scale indices

# Convert indices to actual frequency values
mapped_frequencies = scale[np.clip(np.round(mapped_indices).astype(int), 0, len(scale) - 1)]

# Dynamic adjustment of pitch and volume factors based on wave shape for the full dataset
wave_amplitude = np.abs(np.sin(mapped_frequencies / max(scale)))  # Normalize wave shape
pitch_factor = wave_amplitude / np.max(wave_amplitude)
volume_factor = 0.5 + 0.5 * wave_amplitude  # Adjust as needed for desired volume

# Convert volume_factor to a NumPy array
volume_factor = np.array(volume_factor)

# Plot the mapped frequencies with dynamically adjusted volume factors for the full dataset
plt.figure(figsize=(12, 6))
plt.scatter(data.index, mapped_frequencies, c=volume_factor, cmap='viridis', s=10)
plt.colorbar(label='Volume')
plt.plot(data.index, mapped_frequencies, color='gray', linestyle='-', linewidth=0.5)
plt.title('Mapped Frequencies with Dynamically Adjusted Volume (Full Dataset)')
plt.xlabel('Time')
plt.ylabel('Frequency (Hz)')

# Save the plot as a PNG file
plt.savefig('plots/scatter_mapped_frequencies.png', format='png', dpi=300)  # Adjust dpi as needed
plt.close()  # Close the figure to free memory

# Generate synth audio data based on mapped frequencies of the full dataset
duration = 30  # Fixed duration in seconds for the audio
sample_rate = 44100  # Sample rate for audio
time_values = np.linspace(0, duration, int(sample_rate * duration))

print("Generating synth audio based on C major scale with volume adjustments for the full dataset...")
audio_data = np.zeros(int(sample_rate * duration))
with tqdm(total=len(mapped_frequencies), desc="Synth Audio Progress") as pbar:
    for i, freq in enumerate(mapped_frequencies):
        audio_data += volume_factor[i % len(volume_factor)] * np.sin(2 * np.pi * freq * time_values)
        pbar.update(1)

# Normalize the synth audio data of the full dataset
audio_data /= np.max(np.abs(audio_data))

# Save the synth audio data of the full dataset as a WAV file
write('synth_vibes.wav', sample_rate, audio_data.astype(np.float32))
print("\nSynth audio file generated successfully!")

# Generate revised audio data based on mapped frequencies and volume adjustments
print("Generating wave mapping audio based on mapped frequencies and volume adjustments...")
revised_audio_data = np.zeros(int(sample_rate * duration))

# Create the revised audio signal with progress bar
with tqdm(total=len(mapped_frequencies), desc="Wave Mapping Audio Progress") as pbar:
    for i, freq in enumerate(mapped_frequencies):
        start_time = int(i * (duration / len(mapped_frequencies)) * sample_rate)
        t = np.linspace(0, duration / len(mapped_frequencies), 
                       int(sample_rate * (duration / len(mapped_frequencies))), endpoint=False)
        revised_audio_data[start_time:start_time + len(t)] += volume_factor[i % len(volume_factor)] * np.sin(2 * np.pi * freq * t)
        pbar.update(1)

# Normalize the revised audio data
revised_audio_data /= np.max(np.abs(revised_audio_data))

# Save the revised audio data of the full dataset as a WAV file
write('wave_mapping.wav', sample_rate, revised_audio_data.astype(np.float32))
print("\nWave mapping audio file generated successfully!")

# Generate ECG-like heart sound based on revised audio data
print("Generating ECG-like heart sound based on revised audio data...")
ecg_heartbeat_audio_data = np.zeros(int(sample_rate * duration))

# Create the ECG-like heartbeat audio signal with progress bar
with tqdm(total=len(mapped_frequencies), desc="ECG Heartbeat Audio Progress") as pbar:
    for i, freq in enumerate(mapped_frequencies):
        start_time = int(i * (duration / len(mapped_frequencies)) * sample_rate)
        t = np.linspace(0, duration / len(mapped_frequencies), 
                       int(sample_rate * (duration / len(mapped_frequencies))), endpoint=False)

        # Lower frequency for a bass-like sound
        base_freq = freq * 0.6  # Lower frequency to simulate heartbeat
        heartbeat_freq = 1.0  # Heartbeat pulse frequency (1 Hz for 60 BPM)

        # Create a square wave for the heartbeat effect
        pulse = 0.5 * (1 + np.sign(np.sin(2 * np.pi * heartbeat_freq * t)))  # Square wave pulse

        # Combine the base frequency and the pulse for the heartbeat effect
        ecg_heartbeat_audio_data[start_time:start_time + len(t)] += (
            pulse * 0.5 * np.sin(2 * np.pi * base_freq * t)  # Multiply pulse with the sine wave
        )
        pbar.update(1)

# Normalize the ECG heartbeat audio data
max_value = np.max(np.abs(ecg_heartbeat_audio_data))
if max_value > 0:
    ecg_heartbeat_audio_data /= max_value

# Save the ECG heartbeat audio data of the full dataset as a WAV file
write('ecg_heartbeat.wav', sample_rate, ecg_heartbeat_audio_data.astype(np.float32))
print("\nECG heartbeat audio file generated successfully!")

# Generate spectrograms for all audio files
def generate_spectrogram(audio_data, filename):
    frequencies, times, Sxx = spectrogram(audio_data, sample_rate)
    plt.figure(figsize=(12, 6))
    plt.pcolormesh(times, frequencies, 10 * np.log10(Sxx), shading='gouraud')
    plt.colorbar(label='Intensity (dB)')
    plt.title('Spectrogram of Audio Sound')
    plt.ylabel('Frequency (Hz)')
    plt.xlabel('Time (s)')
    plt.savefig(filename, format='png', dpi=300)
    plt.close()

# Generate spectrograms for all audio files
generate_spectrogram(audio_data, 'plots/spectrogram_synth.png')
generate_spectrogram(revised_audio_data, 'plots/spectrogram_wave_mapping.png')
generate_spectrogram(ecg_heartbeat_audio_data, 'plots/spectrogram_ecg_heartbeat.png')